# BIND2 Redshift — Paper Figures

The **same** conditioned model as a function of scale factor $a=1/(1+z)$ (`redshift_evo.npz`). Truth-validated patch-level multi-z comparison lives in `analysis_redshift.ipynb`; z>0 thermo absolute amplitudes are not yet truth-validated (see WORKLOG).

In [ ]:

import sys
sys.path.insert(0, '/mnt/home/mlee1/vdm_bind2/tools/paper_cache')
import os, pickle
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import paper_config as C

CACHE = C.CACHE_DIR
def L(name):
    p = CACHE / name
    if name.endswith('.pkl'):
        return pickle.load(open(p, 'rb'))
    return dict(np.load(p, allow_pickle=False))

FIG_DIR = Path('paper_figures'); FIG_DIR.mkdir(exist_ok=True)
def save_fig(fig, name, ext=('pdf', 'png')):
    for e in ext:
        fig.savefig(FIG_DIR / f'{name}.{e}', dpi=300, bbox_inches='tight')
    print('  saved', name)

try:
    import scienceplots  # noqa: F401
    plt.style.use(['science', 'notebook'])
except Exception:
    pass
# plt.rcParams.update({'font.size': 10, 'font.family': 'serif', 'mathtext.fontset': 'cm',
#                      'figure.dpi': 110, 'savefig.dpi': 300, 'axes.grid': False})

SUITE_COLORS = C.SUITE_COLORS; SUITE_DISPLAY = C.SUITE_DISPLAY
MASS_CH = C.MASS_CHANNELS; CH_DISPLAY = C.CH_DISPLAY
BIN_LABELS = C.MASS_BIN_LABELS; N_BINS = C.N_MASS_BINS
PARAM_LABELS = C.PARAM_LABELS
# Trained-regime restriction: the paper only uses halos with M200c >= 1e13 (the
# training cut). Lower bins exist in the cache but are never plotted.
MIN_LOG_M200 = 13.0
BINS = [b for b in range(N_BINS) if C.MASS_EDGES[b] >= MIN_LOG_M200]
BIN_CMAP = np.zeros((N_BINS, 4))
BIN_CMAP[BINS] = plt.cm.viridis(np.linspace(0.1, 0.9, len(BINS)))
print('Cache :', CACHE)
print('Model :', C.MODEL_TAG, '| suites', {s: 0 for s in C.SUITES})
print('Files :', sorted(p.name for p in CACHE.glob("*.pkl")) + sorted(p.name for p in CACHE.glob("*.npz")))


## Redshift response of one group ($z=0,0.5,1,2$)

In [ ]:

rz = L('redshift_evo.npz'); z = rz['z_levels']; gz = rz['gz']   # (nz,7,128,128)
rows = [('Gas',1,False),('compton_y',3,True),('T',4,True)]
def lg(im): p=im[im>0]; return np.log10(np.clip(im, p.min() if len(p) else 1e-30, None))
fig, axes = plt.subplots(len(rows), len(z), figsize=(3*len(z), 3*len(rows)))
for r,(lab,ch,_) in enumerate(rows):
    for c in range(len(z)):
        axes[r,c].imshow(lg(gz[c,ch]), cmap='magma'); axes[r,c].set_xticks([]); axes[r,c].set_yticks([])
        if r==0: axes[r,c].set_title(f'z = {z[c]:.1f}')
        if c==0: axes[r,c].set_ylabel(lab)
fig.suptitle(f"fm_redshift · group logM={float(rz['hero_logM']):.2f} · redshift response", y=1.0)
save_fig(fig, 'figR1_redshift_response'); plt.show()


## Amplitude evolution with redshift (BIND, stacked groups)

In [ ]:

rz = L('redshift_evo.npz'); z = rz['z_levels']; amp = rz['amp']  # (nz,3): Gas mass, mean y, mean T
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax,(k,lab) in zip(axes, [(0,'Gas patch mass'),(1,r'mean Compton-$y$'),(2,'mean $T$')]):
    ax.plot(z, amp[:,k]/amp[0,k], 'o-', color='tab:orange'); ax.set_xlabel('$z$'); ax.set_ylabel(f'{lab} (norm. to z=0)'); ax.set_title(lab); ax.grid(alpha=0.3)
fig.suptitle(f"BIND redshift evolution (mean over {int(rz['n_stack'])} groups)")
save_fig(fig, 'figR2_amplitude_evolution'); plt.show()
